# Spotify Data Cleaning: Artist Master

* Simple pandas cleaning for `artists.csv`. 
* The final DataFrame is `clean_artists`. 
* No intermediate CSV export. 
* The last cell loads into MySQL staging.
* Run cells in order. Original CSV goes in `data/raw/`.

---

## 1. Import and read source

In [1]:
import pandas as pd
from pathlib import Path

In [2]:
# Find the source file from the project folder.
def locate_raw_data(filename):
    for folder in (Path.cwd(), *Path.cwd().parents):
        candidate = folder / "data" / "raw" / filename
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(f"Missing CSV in data/raw/: {filename}")

artists_df = pd.read_csv(locate_raw_data("artists.csv"))
print("Source rows:", len(artists_df))

Source rows: 74303


---

## 2. Standardize source columns

In [3]:
# Keep the original artist columns and documented artist-name corrections.
artists_df["artist_name"] = artists_df["artist_name"].astype("string").str.strip()
artists_df["artist_uri"] = (artists_df["artist_uri"].astype("string")
    .str.strip().str.replace(r"^spotify:artist:", "", regex=True))

# Preserve the explicitly documented source corrections for these IDs.
# Preserve the original notebook’s three documented artist-name overrides.
for uri in ["4oPYazJJ1o4rWBrTw9lm40", "0TssKMrMlrXSxIwNrghB1V", "7jXUQgHvIWbra34WkJcezO"]:
    if uri == "7jXUQgHvIWbra34WkJcezO":
        artists_df.loc[artists_df["artist_uri"] == uri, "artist_name"] = pd.NA
    else:
        artists_df.loc[artists_df["artist_uri"] == uri, "artist_name"] = pd.NA

# Keep numeric unknowns as nullable integers instead of invented zeros/ranks.
# Preserve missing listener and rank metrics as null, not fabricated values.
for column in ["monthly_listeners", "monthly_listeners_rank",
               "monthly_listeners_peak_rank", "monthly_listeners_peak_listeners"]:
    if column in artists_df:
        artists_df[column] = pd.to_numeric(artists_df[column], errors="coerce").astype("Int64")


---

## 3. Handle missing values and duplicates

In [4]:
# One observation per chart key; artist master uses one row per artist URI.
keys = ["artist_uri"]
required = ["artist_uri"]
expected_columns = ['artist_uri', 'artist_name', 'monthly_listeners', 'monthly_listeners_rank', 'monthly_listeners_peak_rank', 'monthly_listeners_peak_listeners']
missing_columns = set(expected_columns) - set(artists_df.columns)
if missing_columns:
    raise ValueError(f"Missing staging columns: {sorted(missing_columns)}")

# Standardize blank text. Keep optional unknown measures as SQL NULL.
for col in artists_df.select_dtypes(include=["object", "string"]).columns:
    artists_df[col] = artists_df[col].astype("string").str.strip().replace("", pd.NA)

# Descriptive labels have meaningful defaults; unknown statistics do not.
for col, default in {'artist_name': 'Unknown Artist'}.items():
    artists_df[col] = artists_df[col].fillna(default)

# Invalid optional ranks/counts are unknown, not zero.
for col in ['monthly_listeners_rank', 'monthly_listeners_peak_rank']:
    values = pd.to_numeric(artists_df[col], errors="coerce")
    artists_df[col] = values.where(values > 0).astype("Int64")
for col in ['monthly_listeners', 'monthly_listeners_peak_listeners']:
    values = pd.to_numeric(artists_df[col], errors="coerce")
    artists_df[col] = values.where(values >= 0).astype("Int64")

valid = artists_df[required].notna().all(axis=1)

invalid_count = int((~valid).sum())
artists_df = artists_df.loc[valid]

# Fast single-pass grain deduplication. Keep first original source row.
duplicate_count = int(artists_df.duplicated(subset=keys).sum())
clean_artists = artists_df.drop_duplicates(subset=keys, keep="first").reset_index(drop=True)

print("Invalid key/rank rows removed:", invalid_count)
print("Repeated business keys removed:", duplicate_count)
print("Final staging rows:", len(clean_artists))


Invalid key/rank rows removed: 0
Repeated business keys removed: 0
Final staging rows: 74303


---

## 4. Review and load to MySQL

In [5]:
print("DataFrame:", "clean_artists")
print("Rows:", len(clean_artists))
print("Missing values (optional columns):")
print(clean_artists.isna().sum().to_string())
display(clean_artists.head(5))

DataFrame: clean_artists
Rows: 74303
Missing values (optional columns):
artist_uri                              0
artist_name                             0
monthly_listeners                   49444
monthly_listeners_rank              49444
monthly_listeners_peak_rank         71803
monthly_listeners_peak_listeners    49444


,artist_uri,artist_name,monthly_listeners,monthly_listeners_rank,monthly_listeners_peak_rank,monthly_listeners_peak_listeners
0,5csAaiFcC1qKkVPgUJfXID,Zulu Queens,<NA>,<NA>,<NA>,<NA>
1,2h9NlcdGxIEnnh5aQuSeZ1,Lwah Ndlunkulu,1185698,12533,<NA>,1236495
2,5kwZB1xNLgB3lln1JGS2B9,MaWhoo,2775286,5774,<NA>,2868014
3,1UwTwoC4T1i6vzwsQgIWB0,Feza,1673060,9285,<NA>,1885686
4,6Wqpl1YTjgy5TiTluzFzjx,Tman Xpress,1720092,9030,<NA>,1811260


In [6]:
# Import the shared staging loader from etl/load.py.
import sys
project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "etl" / "load.py").is_file()), None)
if project_root is None:
    raise FileNotFoundError("Missing etl/load.py in the project folder")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
from etl.load import load_dataframe

In [7]:
# Load only the cleaned DataFrame; no exclusion-triggered ValueError.
if clean_artists.empty:
    raise ValueError("No valid records available for staging.")
load_dataframe(clean_artists, "clean_artists")
print("Loaded:", "clean_artists", "| Rows:", len(clean_artists))

Loaded 74,303 rows into spotify_analytics_stg.clean_artists
Loaded: clean_artists | Rows: 74303
